# TinyLlama 1.1B — Portfolio Agent Fine-tune

LoRA fine-tune TinyLlama on pesnik.dev portfolio data.
Works on Colab free T4 (16GB VRAM) with any TRL version.
Auto-detects TRL API and adapts.

## 1. Install, clone & build dataset

Run this cell once. It installs deps, clones the repo, and generates the training dataset.

In [ ]:
import torch, subprocess, os, sys
from pathlib import Path

os.environ["GIT_TERMINAL_PROMPT"] = "0"

# GPU check
device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU"
print(f"GPU: {device_name}")
props = torch.cuda.get_device_properties(0) if torch.cuda.is_available() else None
if props:
    mem_gb = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"VRAM: {mem_gb:.1f}GB")

# Clone repo (public, no auth)
if not Path("pesnik.dev").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pesnik/pesnik.dev.git"], check=True)
os.chdir("pesnik.dev")

# Build dataset
subprocess.run(["python", "scripts/prepare_dataset.py", "--output", "data/train.jsonl"], check=True)

# Install deps (no TRL pin — code handles both old and new APIs)
subprocess.run(["pip", "install", "-qU", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes", "huggingface_hub"], check=True)
print("\nSetup complete. Run the train cell below.")

## 2. Train

Auto-detects your TRL version and uses the correct API.
~20 min on a T4.

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "pesnik-tinyllama-lora"

# ── Dataset ───────────────────────────────────────────────────────────────
dataset = load_dataset("json", data_files="data/train.jsonl", split="train")
print(f"Dataset: {len(dataset)} samples")

# ── Tokenizer ─────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── 4-bit model ───────────────────────────────────────────────────────────
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb,
    device_map="auto", torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
model.gradient_checkpointing_enable()

# ── LoRA ──────────────────────────────────────────────────────────────────
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    bias="none", task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

# ── Detect TRL version and build trainer ─────────────────────────────────
import trl
HAS_SFT_CONFIG = hasattr(trl, "SFTConfig")
print(f"TRL {trl.__version__}  |  SFTConfig available: {HAS_SFT_CONFIG}")

train_kwargs = dict(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    gradient_checkpointing=True,
    num_train_epochs=3,
    learning_rate=2e-4,
    warmup_steps=20,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True, fp16=False,
    dataloader_num_workers=2,
    report_to="none",
    remove_unused_columns=False,
)

if HAS_SFT_CONFIG:
    # Modern TRL (>=0.18): SFTConfig + processing_class
    from trl import SFTConfig, SFTTrainer
    train_kwargs["max_length"] = 1024
    train_kwargs["dataset_text_field"] = "messages"
    train_kwargs["packing"] = False
    args = SFTConfig(**train_kwargs)
    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=dataset,
        processing_class=tokenizer,
    )
else:
    # Legacy TRL (<=0.16): direct kwargs
    from trl import SFTTrainer
    args = TrainingArguments(**train_kwargs)
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=args,
        max_seq_length=1024,
        dataset_text_field="messages",
        packing=False,
    )

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nModel saved to {OUTPUT_DIR}")

## 3. Test inference

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16,
)
model = PeftModel.from_pretrained(base, OUTPUT_DIR)

def ask(question):
    prompt = f"<|user|>\n{question}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.7)
    return tokenizer.decode(out[0], skip_special_tokens=True).split("<|assistant|>\n")[-1].strip()

for q in ["Who are you?", "What's your spirit animal?", "Tell me about Zygote."]:
    print(f"Q: {q}\nA: {ask(q)}\n")

## 4. Push to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi, notebook_login
notebook_login()

api = HfApi()
api.create_repo("pesnik/portfolio-agent", exist_ok=True)
api.upload_folder(
    folder_path=OUTPUT_DIR,
    repo_id="pesnik/portfolio-agent",
    repo_type="model",
)
print("Pushed to https://huggingface.co/pesnik/portfolio-agent")